# 🛡️ Drishti-Kavach: YOLO11m Railway Obstacle Detection Training (Kaggle)

This notebook trains **YOLO11m** (2D Bounding Box Object Detector) on the **Unified 8-Class Railway Obstacle Detection (Day + 850nm NIR Night)** dataset.

### 8 Specialized Railway Obstacle Classes:
0. `Person` (Pedestrians, Trespassers, Track Workers)
1. `Car` (Passenger vehicles at level crossings)
2. `Truck` (Heavy commercial vehicles, buses, tractors)
3. `Branch` (Fallen trees & storm foliage debris)
4. `IronRod` (Deliberate sabotage steel rods, rails, girders)
5. `Boulder` (Landslide rockfalls & placed stones)
6. `Barrel` (Oil/chemical drums placed on track)
7. `Jerrycan` (Flammable fuel canisters & sabotage containers)

*(Note: Trains and 'on-rails' vehicles are strictly excluded to avoid false alarms).*


In [ ]:
# Cell 1: Environment & GPU Verification
import os, sys, glob, shutil, yaml
import torch
import ultralytics
from ultralytics import YOLO

print('[+] Ultralytics Version:', ultralytics.__version__)
print('[+] PyTorch Version:    ', torch.__version__)
print('[+] CUDA Available:     ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('[+] GPU Name:          ', torch.cuda.get_device_name(0))
    print('[+] GPU Count:         ', torch.cuda.device_count())
    print(f'[+] VRAM:               {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# Cell 2: Dataset Discovery & Extraction
input_roots = glob.glob('/kaggle/input/**/dataset_detection', recursive=True)
zip_files = glob.glob('/kaggle/input/**/*.zip', recursive=True)

DATASET_DIR = None
if input_roots:
    DATASET_DIR = input_roots[0]
    print(f'[+] Found dataset directly at: {DATASET_DIR}')
elif zip_files:
    print(f'[+] Found zip archive: {zip_files[0]}. Extracting to /kaggle/temp/...')
    import zipfile
    os.makedirs('/kaggle/temp', exist_ok=True)
    with zipfile.ZipFile(zip_files[0], 'r') as zf:
        zf.extractall('/kaggle/temp')
    found = glob.glob('/kaggle/temp/**/dataset_detection', recursive=True)
    DATASET_DIR = found[0] if found else '/kaggle/temp/dataset_detection'
else:
    DATASET_DIR = 'dataset_detection'

train_imgs = glob.glob(f'{DATASET_DIR}/images/train/*.jpg')
val_imgs = glob.glob(f'{DATASET_DIR}/images/val/*.jpg')
print(f'[+] Verified Detection Dataset: {len(train_imgs)} Train Frames | {len(val_imgs)} Val Frames')


In [ ]:
# Cell 4: Train YOLO11m with Transfer Learning
EPOCHS = 30        # Transfer learning converges in 25-30 epochs (~60-75 mins total)
IMGSZ = 1024       # 1024px for small obstacle detection at distance
BATCH_SIZE = 16    # 16 per batch on T4 / P100

print("=" * 75)
print(f" 🛡️ INITIALIZING YOLO11m RAILWAY OBSTACLE DETECTOR ({EPOCHS} EPOCHS)")
print(f" • Model Base:     yolo11m.pt (COCO Pretrained weights)")
print(f" • Resolution:     {IMGSZ}x{IMGSZ}")
print(f" • Batch Size:     {BATCH_SIZE}")
print(f" • Estimated Time: ~60-75 minutes total (Well within 12-hr limit!)")
print("=" * 75)

model = YOLO("yolo11m.pt")

results = model.train(
    data=kaggle_yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    patience=8,
    device=0,
    workers=4,
    amp=True,
    save=True,
    project="/kaggle/working/runs",
    name="yolo11m_raildrishti",
    exist_ok=True,
    plots=True
)

print("[+] Training completed successfully!")


In [ ]:
# Cell 4: Train YOLO11m with Transfer Learning
EPOCHS = 50
IMGSZ = 1024       # 1024px for small sabotage item detection at distance
BATCH_SIZE = 16    # 16 per batch on T4 / P100

print('=' * 75)
print(f' 🛡️ INITIALIZING YOLO11m RAILWAY OBSTACLE DETECTOR ({EPOCHS} EPOCHS)')
print(f' • Model Base:     yolo11m.pt (Pretrained weights)')
print(f' • Resolution:     {IMGSZ}x{IMGSZ}')
print(f' • Batch Size:     {BATCH_SIZE}')
print('=' * 75)

model = YOLO('yolo11m.pt')

results = model.train(
    data=kaggle_yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    device=0,
    amp=True,
    save=True,
    project='/kaggle/working/runs',
    name='yolo11m_raildrishti',
    exist_ok=True,
    plots=True
)

print('[+] Training completed successfully!')


In [ ]:
# Cell 5: Validation & Metrics Summary
print('[*] Running Final Validation Evaluation...')
best_pt_path = '/kaggle/working/runs/yolo11m_raildrishti/weights/best.pt'

if os.path.exists(best_pt_path):
    shutil.copy(best_pt_path, '/kaggle/working/best_yolo11m_raildrishti.pt')
    print('[+] Copied best weights to: /kaggle/working/best_yolo11m_raildrishti.pt')
    
    best_model = YOLO('/kaggle/working/best_yolo11m_raildrishti.pt')
    val_results = best_model.val(data=kaggle_yaml_path, imgsz=1024, split='val')
    print('\n📊 Overall Metrics:')
    print(f' • mAP50:    {val_results.box.map50*100:.2f}%')
    print(f' • mAP50-95: {val_results.box.map*100:.2f}%')
    print(f' • Precision: {val_results.box.mp*100:.2f}%')
    print(f' • Recall:    {val_results.box.mr*100:.2f}%')


In [ ]:
# Cell 6: Export Best Model to ONNX
print('[*] Exporting Best YOLO11m Model to ONNX format...')
if os.path.exists('/kaggle/working/best_yolo11m_raildrishti.pt'):
    export_model = YOLO('/kaggle/working/best_yolo11m_raildrishti.pt')
    onnx_path = export_model.export(format='onnx', imgsz=1024, dynamic=True, opset=14)
    print(f'[+] Successfully exported: {onnx_path}')
